# Data Cleansing

In [ ]:
import pandas as pd
import geopy.geocoders
from geopy.geocoders import Nominatim
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
import geopandas as gpd
import os
import numpy as np
from math import floor
from multiprocess import Pool, cpu_count
import os
import re
import pandas as pd

In [ ]:
parent_path = "/Users/krishna/Rutgers/DataScience/Project/final_project/csv_files"

In [ ]:
nypd_path = 'nypd_11_19.csv'
nypd_dataset = pd.read_csv(os.path.join(parent_path, nypd_path))

In [ ]:
file_loc = "ny/ny.shp"
fp = os.path.join(parent_path, file_loc)
data = gpd.read_file(fp)

In [ ]:
max = nypd_dataset.shape[0]
lim = round(max/cpu_count())
data_range = []
for i in range(0,cpu_count()):
    if i == (cpu_count()-1):
        data_range.append((lim*i, max))
    else:
        data_range.append((lim*i, lim*(i+1)))

# Responsible for identifying the zipcode using the longitude and latitude

In [ ]:
def get_zip_code(data_range):
    zipcodes = []
    for index, rows in nypd_dataset.iterrows():
        is_found = False
        if index < data_range[0]:
            continue
        if index > data_range[1] - 1:
            break

        point = Point((rows['Longitude'], rows['Latitude']))
        for itr, geoms in enumerate(data['geometry']):
            if geoms.intersects(point):
                zipcodes.append(data["ZCTA5CE20"][itr])
                is_found = True
                break

        
        if not is_found:
            zipcodes.append(0)
    
    return zipcodes

In [ ]:
zip_codes = []
with Pool(processes=cpu_count()) as p:
    for data in p.map(get_zip_code, data_range):
        zip_codes.extend(data)
        # zipcodes = get_zip_code(10000, 40000)

In [ ]:
print(len(zip_codes), nypd_dataset.shape[0])

In [ ]:
nypd_df = nypd_dataset

In [ ]:
nypd_df["zipcode"] = zip_codes

In [ ]:
nypd_df = nypd_df[nypd_df.zipcode != 0]

In [ ]:
nypd_df

In [ ]:
nypd_df.drop(nypd_df.columns[nypd_df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)

In [ ]:
path = "nypd_final.csv"
nypd_df.to_csv(os.path.join(parent_path, path))

## NYC Populus Deatils
Details for the populus spread across multiple sheets are joint in this.

In [ ]:
files_df = list()

parent_path = "/Users/krishna/Rutgers/DataScience/Project/final_project/csv_files"
child_path = ["income", "pop_employment", "schooling"]
for child in child_path:
    path = os.path.join(parent_path, child)
    for filename in os.listdir(path):
        if re.match(".*Data.csv", filename):
            year = filename.split(".", 1)[0][-4:]
            file_df = pd.read_csv(os.path.join(path, filename))[1:]
            name_zipcodes = []
            for val in file_df["NAME"]:
                name_zipcodes.append(val[-5:])

            file_df["year"] = year
            file_df["zipcode"] = name_zipcodes
            files_df.append(file_df)
    
    final_df = pd.concat(files_df)
    output_path = os.path.join(parent_path, child + ".csv")
    final_df.drop(final_df.columns[final_df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)
    final_df.to_csv(output_path)
    files_df.clear()